# Prerequisite 07 · Gymnasium：环境交互契约

[上一章：卷积网络与序列模型](Prerequisite06B_CNN_and_Sequences.ipynb) · [下一章：从多臂老虎机走进 RL](Prerequisite07B_Bandit_Bridge.ipynb)

**目标：**正确使用 `make/reset/step`，理解 spaces、episode、`terminated/truncated`、seed 和 wrapper。Gymnasium 是接口，不是强化学习算法。


## 1. 最小随机 agent

`reset()` 返回 observation 和 info；`step()` 返回五项。`terminated` 表示任务 MDP 的终止，`truncated` 通常表示时间限制或外部截断。


In [1]:
import gymnasium as gym

env = gym.make("CartPole-v1")
observation, info = env.reset(seed=7)
total_reward = 0.0
for step in range(500):
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if terminated or truncated:
        break
env.close()
print("observation shape:", observation.shape)
print("episode steps/return/end flags:", step+1, total_reward, terminated, truncated)


observation shape: (4,)
episode steps/return/end flags: 16 16.0 True False


## 2. Spaces 是数据契约

- `Discrete(n)`：整数动作 `0..n-1`。
- `Box(low, high, shape)`：连续向量或图像。
- observation/action 必须满足对应 space 的 shape、dtype 与范围。

wrapper 可以变换 observation、action 或 reward，但必须同步更新 space，并保持 reset/step 契约。


In [2]:
env = gym.make("CartPole-v1")
print("observation space:", env.observation_space)
print("action space:", env.action_space)
obs, _ = env.reset(seed=123)
assert env.observation_space.contains(obs)
assert env.action_space.contains(0)
env.close()


observation space: Box([-4.8000002e+00 -3.4028235e+38 -4.1887903e-01 -3.4028235e+38], [4.8000002e+00 3.4028235e+38 4.1887903e-01 3.4028235e+38], (4,), float32)
action space: Discrete(2)


## 3. 结束 episode 与停止 bootstrap 不是同一问题

采样循环通常在 `terminated or truncated` 时 reset。但构造价值 target 时：

- `terminated=True`：任务真正结束，通常不 bootstrap。
- `truncated=True`：只是时间限制，若下一状态仍有价值，通常应该 bootstrap。

具体仍要检查环境语义和数据采集方式；不要把旧教程中的单一 `done` 无脑复制到新 API。


In [3]:
def td_target(reward, next_value, gamma, terminated):
    return reward if terminated else reward + gamma*next_value

print("true terminal target:", td_target(1.0, 10.0, .9, True))
print("time-limit target:", td_target(1.0, 10.0, .9, False))
assert td_target(1.0, 10.0, .9, True) == 1.0


true terminal target: 1.0
time-limit target: 10.0


## 练习与过关标准

**练习：**运行 20 个随机 CartPole episode，记录 return 的均值和标准差；分别统计 `terminated` 与 `truncated` 次数。

**过关：**能写完整 episode loop；会检查 space；能解释 reset 的 seed；能区分 episode reset 条件与 bootstrap mask。

**官方选读：**[Gymnasium Basic Usage](https://gymnasium.farama.org/introduction/basic_usage/)；遇到旧版 `done` 代码时读 [Gymnasium Migration Guide](https://gymnasium.farama.org/main/introduction/migration_guide/)。
